# ISLES24 Clinical Data Explorer

Interactive exploration of the clinical summary workbook for ISLES24.

The workbook contains aggregate statistics stratified by center and by train/test
subset. This notebook reconstructs visual summaries from those aggregate values.

**Important:** the workbook does not contain individual patient-level observations.
Therefore, continuous-variable boxplots are reconstructed from the reported
quartiles/median and selected reported fences. They are not exact raw-data boxplots.

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    widgets = None

pd.set_option("display.max_columns", 50)

In [ ]:
def find_workbook():
    env_path = os.environ.get("ISLES24_SUMMARY_XLSX")
    if env_path and Path(env_path).exists():
        return Path(env_path)

    candidates = [
        Path("isles24_summary.xlsx"),
        Path("../isles24_summary.xlsx"),
        Path("data/isles24_summary.xlsx"),
        Path("../data/isles24_summary.xlsx"),
        Path("utils/isles24_summary.xlsx"),
        Path("../utils/isles24_summary.xlsx"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not find isles24_summary.xlsx. "
        "Put it in the project root or set ISLES24_SUMMARY_XLSX."
    )

workbook_path = find_workbook()
print(f"Using workbook: {workbook_path}")

In [ ]:
SHEETS = [
    "All Data",
    "Stratified (Train vs Test)",
    "Stratified (Center 1 vs 2)",
]

xls = pd.ExcelFile(workbook_path)
print("Available sheets:", xls.sheet_names)

data = {}

for sheet in SHEETS:
    df = pd.read_excel(workbook_path, sheet_name=sheet)
    df.columns = [str(c).strip() for c in df.columns]

    # The fourth column is the grouping column in the supplied workbook.
    if "Group" not in df.columns and len(df.columns) >= 4:
        df = df.rename(columns={df.columns[3]: "Group"})

    data[sheet] = df
    print(f"{sheet}: {df.shape}")

In [ ]:
def as_number(value):
    if pd.isna(value) or str(value).strip() == "-":
        return np.nan
    return pd.to_numeric(value, errors="coerce")


FREQ_RE = re.compile(
    r"(?:^|,\s*)(.*?):\s*(\d+)\s*\(([\d.]+)%\)"
    r"(?=,\s*.*?:\s*\d+\s*\(|$)"
)


def parse_frequency(value):
    if pd.isna(value):
        return []

    text = str(value).strip()
    result = []

    for label, count, pct in FREQ_RE.findall(text):
        result.append(
            {
                "Label": label.strip(),
                "Count": int(count),
                "Percent": float(pct),
            }
        )

    return result


def variable_kind(df):
    if "Median" in df.columns:
        numeric_values = pd.to_numeric(df["Median"], errors="coerce")
        if numeric_values.notna().any():
            return "continuous"

    if "Frequency" in df.columns:
        if df["Frequency"].apply(parse_frequency).map(len).max() > 0:
            return "categorical"

    return "unknown"


def reported_test_text(rows):
    if rows.empty:
        return ""

    tests = []
    if "Statistical Test" in rows.columns:
        tests = [
            str(x).strip()
            for x in rows["Statistical Test"].dropna().unique()
            if str(x).strip() and str(x).strip() != "-"
        ]

    pvals = []
    if "p-value" in rows.columns:
        pvals = [
            str(x).strip()
            for x in rows["p-value"].dropna().unique()
            if str(x).strip() and str(x).strip() != "-"
        ]

    text = ""
    if tests:
        text += f"Test: {tests[0]}"
    if pvals:
        text += f" | p-value: {pvals[0]}"

    return text

In [ ]:
def plot_continuous(rows, variable, whisker_mode="5th/95th percentiles"):
    rows = rows.copy()

    rows["Median_num"] = rows["Median"].apply(as_number)
    rows["Q1_num"] = rows["25th Percentile (Q1)"].apply(as_number)
    rows["Q3_num"] = rows["75th Percentile (Q3)"].apply(as_number)

    if whisker_mode == "Min/max":
        lower_col = "Min"
        upper_col = "Max"
    else:
        lower_col = "5th Percentile"
        upper_col = "95th Percentile"

    rows["Lower_num"] = rows[lower_col].apply(as_number)
    rows["Upper_num"] = rows[upper_col].apply(as_number)

    # Fall back to Q1/Q3 if a selected fence is unavailable.
    rows["Lower_num"] = rows["Lower_num"].fillna(rows["Q1_num"])
    rows["Upper_num"] = rows["Upper_num"].fillna(rows["Q3_num"])

    rows = rows.dropna(subset=["Median_num", "Q1_num", "Q3_num"])

    fig = go.Figure()

    for _, row in rows.iterrows():
        group = str(row["Group"])

        customdata = [[
            row.get("Missing cases", np.nan),
            row.get("Mean", np.nan),
            row.get("IQR", np.nan),
        ]]

        fig.add_trace(
            go.Box(
                name=group,
                q1=[row["Q1_num"]],
                median=[row["Median_num"]],
                q3=[row["Q3_num"]],
                lowerfence=[row["Lower_num"]],
                upperfence=[row["Upper_num"]],
                mean=[as_number(row.get("Mean", np.nan))],
                boxpoints=False,
                boxmean=True,
                customdata=customdata,
                hovertemplate=(
                    "<b>%{x}</b><br>"
                    "Q1: %{q1}<br>"
                    "Median: %{median}<br>"
                    "Q3: %{q3}<br>"
                    "Lower fence: %{lowerfence}<br>"
                    "Upper fence: %{upperfence}<br>"
                    "Mean: %{mean}<br>"
                    "Missing cases: %{customdata[0]}<br>"
                    "IQR: %{customdata[2]}<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title=f"{variable} — reconstructed boxplot",
        yaxis_title=variable,
        xaxis_title="Group",
        template="plotly_white",
    )

    return fig


def plot_categorical(rows, variable, percentage=True):
    fig = go.Figure()

    for _, row in rows.iterrows():
        group = str(row["Group"])
        parsed = parse_frequency(row.get("Frequency"))

        if not parsed:
            continue

        labels = [item["Label"] for item in parsed]
        values = [
            item["Percent"] if percentage else item["Count"]
            for item in parsed
        ]

        fig.add_trace(
            go.Bar(
                name=group,
                x=labels,
                y=values,
                hovertemplate=(
                    "<b>%{x}</b><br>"
                    + ("Percent: %{y:.1f}%" if percentage else "Count: %{y}")
                    + "<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        barmode="group",
        title=f"{variable} — categorical distribution",
        xaxis_title=variable,
        yaxis_title="Percent (%)" if percentage else "Count",
        template="plotly_white",
    )

    return fig


def plot_variable(rows, variable, whisker_mode="5th/95th percentiles",
                  percentage=True):
    kind = variable_kind(rows)

    if kind == "continuous":
        return plot_continuous(rows, variable, whisker_mode)

    if kind == "categorical":
        return plot_categorical(rows, variable, percentage)

    raise ValueError(f"Could not determine variable type for: {variable}")

## Interactive explorer

Use the controls below to select the comparison, category, timepoint and variable.
For continuous variables, you can choose whether the reconstructed whiskers use
the reported minimum/maximum or the reported 5th/95th percentiles.

In [ ]:
if widgets is None:
    print(
        "ipywidgets is not installed. "
        "Install it with: pip install ipywidgets"
    )
else:
    comparison_widget = widgets.Dropdown(
        options=SHEETS,
        value=SHEETS[1],
        description="Compare:",
        layout=widgets.Layout(width="500px"),
    )

    category_widget = widgets.Dropdown(
        options=[],
        description="Category:",
        layout=widgets.Layout(width="500px"),
    )

    timepoint_widget = widgets.Dropdown(
        options=[],
        description="Timepoint:",
        layout=widgets.Layout(width="500px"),
    )

    variable_widget = widgets.Dropdown(
        options=[],
        description="Variable:",
        layout=widgets.Layout(width="500px"),
    )

    whisker_widget = widgets.Dropdown(
        options=["5th/95th percentiles", "Min/max"],
        description="Whiskers:",
        layout=widgets.Layout(width="500px"),
    )

    percentage_widget = widgets.Checkbox(
        value=True,
        description="Show percentages for categorical variables",
    )

    output = widgets.Output()

    def refresh_categories(*_):
        df = data[comparison_widget.value]
        categories = sorted(
            [x for x in df["Category"].dropna().unique()]
        )

        category_widget.options = categories

        if categories:
            category_widget.value = categories[0]
        else:
            category_widget.options = [""]

    def refresh_timepoints(*_):
        df = data[comparison_widget.value]
        filtered = df[df["Category"] == category_widget.value]

        timepoints = sorted(
            [x for x in filtered["Timepoint"].dropna().unique()]
        )

        timepoint_widget.options = timepoints

        if timepoints:
            timepoint_widget.value = timepoints[0]

    def refresh_variables(*_):
        df = data[comparison_widget.value]

        filtered = df[
            (df["Category"] == category_widget.value)
            & (df["Timepoint"] == timepoint_widget.value)
        ]

        variables = sorted(
            [x for x in filtered["Variable"].dropna().unique()]
        )

        variable_widget.options = variables

        if variables:
            variable_widget.value = variables[0]

    def draw(*_):
        with output:
            output.clear_output(wait=True)

            df = data[comparison_widget.value]

            rows = df[
                (df["Category"] == category_widget.value)
                & (df["Timepoint"] == timepoint_widget.value)
                & (df["Variable"] == variable_widget.value)
            ].copy()

            if rows.empty:
                print("No data available for this selection.")
                return

            try:
                fig = plot_variable(
                    rows,
                    variable_widget.value,
                    whisker_mode=whisker_widget.value,
                    percentage=percentage_widget.value,
                )
                fig.show()

                test_text = reported_test_text(rows)
                if test_text:
                    print(test_text)

            except Exception as exc:
                print(f"Could not plot this variable: {exc}")

    comparison_widget.observe(refresh_categories, names="value")
    category_widget.observe(refresh_timepoints, names="value")
    timepoint_widget.observe(refresh_variables, names="value")

    for widget in [
        comparison_widget,
        category_widget,
        timepoint_widget,
        variable_widget,
        whisker_widget,
        percentage_widget,
    ]:
        widget.observe(draw, names="value")

    refresh_categories()
    refresh_timepoints()
    refresh_variables()

    display(
        widgets.VBox([
            comparison_widget,
            category_widget,
            timepoint_widget,
            variable_widget,
            whisker_widget,
            percentage_widget,
            output,
        ])
    )

## Programmatic example

The explorer can also be used without the widgets. The following example plots
age for the center comparison, when that variable is available in the workbook.

In [ ]:
example_sheet = "Stratified (Center 1 vs 2)"
example_df = data[example_sheet]

example_rows = example_df[
    example_df["Variable"].astype(str).str.strip().eq("Age (years)")
].copy()

if not example_rows.empty:
    fig = plot_continuous(
        example_rows,
        "Age (years)",
        whisker_mode="5th/95th percentiles",
    )
    fig.show()
else:
    print("Age (years) was not found in the center comparison sheet.")

## Notes for maintainers

- The notebook uses aggregate statistics from `isles24_summary.xlsx`.
- Continuous boxplots use the reported Q1, median and Q3, with either
  min/max or 5th/95th percentile fences.
- Exact raw-data Tukey whiskers and individual observations cannot be reconstructed
  from the summary workbook alone.
- Categorical variables are parsed from the workbook's `Frequency` strings.
- The notebook is intentionally independent of the absolute path on the author's
  computer. It searches common project-relative locations and also supports the
  `ISLES24_SUMMARY_XLSX` environment variable.